### **Razonamiento multimodal, grounding y alucinación visual**

#### **Verificación de evidencias, consistencia y análisis de errores en VLMs**

El objetivo de este cuaderno es evaluar respuestas multimodales no solo por exactitud, sino por evidencia, consistencia, grounding, alucinación y tipo de error.

El cuaderno está diseñado para ejecutarse sin GPU y sin descargar modelos grandes. Usa un protocolo didáctico con datos pequeños, respuestas simuladas y funciones de evaluación reproducibles. Al final se incluye una extensión avanzada para conectar un VLM real y un detector externo.


### **Parte 1: De evaluación a razonamiento grounded**

#### **Idea central**

En evaluación multimodal no basta preguntar si la respuesta final es correcta. También se debe preguntar si la respuesta está apoyada por evidencia presente en la entrada.

Un modelo puede fallar de varias formas:

1. Puede percibir mal la imagen.
2. Puede leer mal texto incluido.
3. Puede contar de forma incorrecta.
4. Puede inventar objetos que no están presentes.
5. Puede responder con seguridad excesiva.
6. Puede dar una explicación fluida pero no verificable.

En este cuaderno llamaremos **grounding** a la relación verificable entre la respuesta del modelo y la evidencia disponible en la entrada multimodal.


In [ ]:
# Este cuaderno evita dependencias pesadas para que pueda ejecutarse en CPU.

from __future__ import annotations

import json
import random
import platform
from dataclasses import dataclass, asdict
from pathlib import Path
from statistics import mean
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

SEED = 42
random.seed(SEED)

OUTPUT_DIR = Path("results") / "cuaderno16_mcc225"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Entorno listo")
print(f"Semilla fijada: {SEED}")
print(f"Directorio de resultados: {OUTPUT_DIR}")


### **Parte 2: Configuración experimental y reproducibilidad**

#### **Metadatos mínimos**

Todo experimento debe dejar registro de la configuración usada. En un cuaderno docente esto no necesita ser complejo, pero sí debe permitir reconstruir qué se hizo.

Se registran:

1. Identificador del experimento.
2. Modelo evaluado o simulador usado.
3. Semilla.
4. Tamaño del conjunto de evaluación.
5. Métricas calculadas.
6. Versión de Python y sistema.


In [ ]:
@dataclass
class ExperimentalConfig:
    """Configuración mínima para reproducibilidad del experimento."""
    experiment_id: str
    researcher: str
    model_name: str
    dataset_name: str
    seed: int
    sample_size: int
    prompt_variants: List[str]
    metrics: List[str]
    python_version: str
    platform_name: str

    def to_dict(self):
        """Convierte la configuración a diccionario serializable."""
        return asdict(self)

    def save(self, output_dir: Path):
        """Guarda la configuración en formato JSON."""
        output_path = output_dir / "configuracion_experimental.json"
        with output_path.open("w", encoding="utf-8") as file:
            json.dump(self.to_dict(), file, indent=2, ensure_ascii=False)
        return output_path


config = ExperimentalConfig(
    experiment_id="mcc225_semana10_grounding_001",
    researcher="Estudiante MCC225",
    model_name="Simulador VLM reproducible",
    dataset_name="Conjunto pequeño de sondeo multimodal",
    seed=SEED,
    sample_size=8,
    prompt_variants=["directo", "con_evidencia", "cauteloso"],
    metrics=[
        "exactitud",
        "tasa_falsos_positivos",
        "tasa_falsos_negativos",
        "tasa_alucinacion",
        "consistencia_por_prompt",
        "severidad_media"
    ],
    python_version=platform.python_version(),
    platform_name=platform.platform()
)

config_path = config.save(OUTPUT_DIR)
print(f"Configuración guardada en: {config_path}")


### **Parte 3: Dataset pequeño de razonamiento multimodal**

#### **Diseño del conjunto de sondeo**

El conjunto de casos no intenta reemplazar un benchmark. Su función es permitir una auditoría controlada de fallas.

Se incluyen casos con:

1. Objetos presentes y ausentes.
2. Texto incluido.
3. Relaciones espaciales.
4. Conteo.
5. Contradicción entre pregunta y escena.
6. Posible alucinación visual.

En un experimento real, cada caso debería estar asociado a una imagen, sus anotaciones y una licencia de uso. Aquí usamos descripciones estructuradas para que el cuaderno sea ejecutable sin archivos externos.


In [ ]:
@dataclass
class GroundingCase:
    """Caso de sondeo con anotaciones verificables."""
    case_id: str
    visual_description: str
    question: str
    expected_answer: str
    expected_objects: List[str]
    absent_objects: List[str]
    expected_evidence: List[str]
    task_type: str
    risk_note: str


cases = [
    GroundingCase(
        case_id="caso_01",
        visual_description="Una mesa con una taza roja, un libro azul y una laptop abierta.",
        question="¿Hay una taza roja sobre la mesa?",
        expected_answer="sí",
        expected_objects=["mesa", "taza roja", "libro azul", "laptop"],
        absent_objects=["perro", "bicicleta", "teléfono"],
        expected_evidence=["taza roja", "mesa"],
        task_type="presencia de objeto",
        risk_note="Riesgo bajo si el error solo afecta una descripción simple."
    ),
    GroundingCase(
        case_id="caso_02",
        visual_description="Una señal dice SALIDA. Debajo de la señal hay una puerta verde.",
        question="¿Qué palabra aparece en la señal?",
        expected_answer="salida",
        expected_objects=["señal", "puerta verde"],
        absent_objects=["entrada", "ascensor", "automóvil"],
        expected_evidence=["señal", "salida"],
        task_type="ocr",
        risk_note="Riesgo medio si la lectura de texto guía una acción."
    ),
    GroundingCase(
        case_id="caso_03",
        visual_description="Un gato está debajo de una silla. Una pelota está encima de la silla.",
        question="¿Dónde está el gato respecto de la silla?",
        expected_answer="debajo",
        expected_objects=["gato", "silla", "pelota"],
        absent_objects=["perro", "mesa", "cama"],
        expected_evidence=["gato", "silla", "debajo"],
        task_type="relación espacial",
        risk_note="Riesgo medio por confusión espacial."
    ),
    GroundingCase(
        case_id="caso_04",
        visual_description="Hay tres botellas transparentes alineadas junto a una mochila negra.",
        question="¿Cuántas botellas hay?",
        expected_answer="tres",
        expected_objects=["botella", "mochila negra"],
        absent_objects=["cuatro botellas", "vaso", "plato"],
        expected_evidence=["tres botellas", "botellas alineadas"],
        task_type="conteo",
        risk_note="Riesgo medio si el conteo se usa para inventario."
    ),
    GroundingCase(
        case_id="caso_05",
        visual_description="Una persona sostiene un paraguas amarillo. No hay lluvia visible.",
        question="¿Está lloviendo en la imagen?",
        expected_answer="no",
        expected_objects=["persona", "paraguas amarillo"],
        absent_objects=["lluvia", "charcos", "nubes oscuras"],
        expected_evidence=["no hay lluvia visible", "paraguas amarillo"],
        task_type="pregunta negativa",
        risk_note="Riesgo alto si el modelo infiere una condición no visible."
    ),
    GroundingCase(
        case_id="caso_06",
        visual_description="Un autobús azul está estacionado frente a un edificio. No aparece ningún tren.",
        question="¿Hay un tren en la escena?",
        expected_answer="no",
        expected_objects=["autobús azul", "edificio"],
        absent_objects=["tren", "vagón", "riel"],
        expected_evidence=["autobús azul", "no aparece ningún tren"],
        task_type="objeto distractor",
        risk_note="Riesgo alto si el modelo confunde objetos visualmente o semánticamente cercanos."
    ),
    GroundingCase(
        case_id="caso_07",
        visual_description="Una receta impresa muestra la palabra AZÚCAR y una cuchara junto al papel.",
        question="¿El texto menciona sal?",
        expected_answer="no",
        expected_objects=["receta", "azúcar", "cuchara"],
        absent_objects=["sal", "pimienta", "aceite"],
        expected_evidence=["azúcar", "no menciona sal"],
        task_type="ocr negativo",
        risk_note="Riesgo alto si el sistema sustituye palabras por asociaciones comunes."
    ),
    GroundingCase(
        case_id="caso_08",
        visual_description="Dos niños juegan con una pelota en un parque. Hay un banco detrás de ellos.",
        question="¿Qué objeto está detrás de los niños?",
        expected_answer="banco",
        expected_objects=["niños", "pelota", "parque", "banco"],
        absent_objects=["auto", "computadora", "semáforo"],
        expected_evidence=["banco detrás de ellos"],
        task_type="relación espacial",
        risk_note="Riesgo medio por dependencia de relaciones espaciales."
    )
]

print(f"Número de casos: {len(cases)}")


In [ ]:
def cases_to_records(cases: Sequence[GroundingCase]):
    """Convierte los casos a registros tabulares."""
    records = []
    for case in cases:
        records.append({
            "id": case.case_id,
            "descripción_visual": case.visual_description,
            "pregunta": case.question,
            "respuesta_esperada": case.expected_answer,
            "tipo_tarea": case.task_type,
            "evidencia_esperada": ", ".join(case.expected_evidence),
            "nota_riesgo": case.risk_note
        })
    return records


case_records = cases_to_records(cases)

if pd is not None:
    display(pd.DataFrame(case_records))
else:
    for record in case_records:
        print(json.dumps(record, ensure_ascii=False, indent=2))


### **Parte 4: Pipeline de inferencia**

#### **Componentes del pipeline**

El pipeline mínimo tiene cinco pasos:

1. Seleccionar un caso.
2. Formular una variante de prompt.
3. Obtener la respuesta cruda del modelo.
4. Normalizar la respuesta.
5. Extraer evidencia declarada por el modelo.

En un sistema real, la función de inferencia llamaría a un VLM. En este cuaderno se usa un simulador para reproducir fallas típicas sin depender de GPU.


In [ ]:
@dataclass
class ModelResponse:
    """Respuesta del sistema para una variante de prompt."""
    case_id: str
    prompt_variant: str
    raw_response: str
    normalized_answer: Optional[str]
    declared_evidence: List[str]
    confidence_label: str


def build_prompt(case: GroundingCase, prompt_variant: str):
    """Construye una variante de prompt para el caso."""
    if prompt_variant == "directo":
        return f"Responde la pregunta de forma breve: {case.question}"
    if prompt_variant == "con_evidencia":
        return f"Responde la pregunta y menciona evidencia visible: {case.question}"
    if prompt_variant == "cauteloso":
        return f"Responde solo si hay evidencia suficiente. Pregunta: {case.question}"
    raise ValueError(f"Variante de prompt no reconocida: {prompt_variant}")


def generate_mock_response(case: GroundingCase, prompt_variant: str):
    """Genera una respuesta simulada con fallas controladas."""
    scripted_errors = {
        ("caso_05", "directo"): "Sí, parece que llueve porque la persona sostiene un paraguas.",
        ("caso_06", "directo"): "Sí, hay un tren cerca del edificio.",
        ("caso_07", "con_evidencia"): "Sí, el texto menciona sal junto a la receta.",
        ("caso_04", "cauteloso"): "Hay cuatro botellas, aunque la imagen no es completamente clara.",
        ("caso_03", "directo"): "El gato está encima de la silla."
    }

    if (case.case_id, prompt_variant) in scripted_errors:
        return scripted_errors[(case.case_id, prompt_variant)]

    if case.expected_answer in ["sí", "no"]:
        if prompt_variant == "con_evidencia":
            return f"{case.expected_answer.capitalize()}. La evidencia visible es: {', '.join(case.expected_evidence)}."
        if prompt_variant == "cauteloso":
            return f"{case.expected_answer.capitalize()}. La respuesta se apoya en la evidencia visible del caso."
        return f"{case.expected_answer.capitalize()}."

    if prompt_variant == "con_evidencia":
        return f"La respuesta es {case.expected_answer}. Evidencia: {', '.join(case.expected_evidence)}."
    if prompt_variant == "cauteloso":
        return f"La respuesta más respaldada por la evidencia es {case.expected_answer}."
    return f"{case.expected_answer.capitalize()}."


def normalize_answer(response: str):
    """Normaliza respuestas frecuentes para comparación básica."""
    text = response.lower().strip()

    if text.startswith("sí") or text.startswith("si,") or text.startswith("sí,"):
        return "sí"
    if text.startswith("no"):
        return "no"

    numeric_map = {
        "uno": "uno",
        "dos": "dos",
        "tres": "tres",
        "cuatro": "cuatro",
        "cinco": "cinco",
    }
    for word, value in numeric_map.items():
        if word in text:
            return value

    candidate_answers = ["salida", "debajo", "banco"]
    for answer in candidate_answers:
        if answer in text:
            return answer

    return None


def extract_declared_evidence(response: str, case: GroundingCase):
    """Extrae evidencia declarada mediante coincidencias simples."""
    text = response.lower()
    evidence = []
    candidates = case.expected_objects + case.absent_objects + case.expected_evidence

    for item in candidates:
        if item.lower() in text:
            evidence.append(item)

    return sorted(set(evidence))


def estimate_confidence_label(response: str):
    """Estima confianza lingüística de forma aproximada."""
    text = response.lower()

    cautious_terms = ["parece", "posible", "no es completamente clara", "más respaldada"]
    strong_terms = ["claramente", "sin duda", "hay", "la respuesta es"]

    if any(term in text for term in cautious_terms):
        return "cautelosa"
    if any(term in text for term in strong_terms):
        return "alta"
    return "media"


In [ ]:
def run_inference(cases: Sequence[GroundingCase], prompt_variants: Sequence[str]):
    """Ejecuta inferencia simulada sobre todos los casos y variantes."""
    responses = []

    for case in cases:
        for prompt_variant in prompt_variants:
            _ = build_prompt(case, prompt_variant)
            raw_response = generate_mock_response(case, prompt_variant)
            normalized_answer = normalize_answer(raw_response)
            declared_evidence = extract_declared_evidence(raw_response, case)
            confidence_label = estimate_confidence_label(raw_response)

            responses.append(ModelResponse(
                case_id=case.case_id,
                prompt_variant=prompt_variant,
                raw_response=raw_response,
                normalized_answer=normalized_answer,
                declared_evidence=declared_evidence,
                confidence_label=confidence_label
            ))

    return responses


responses = run_inference(cases, config.prompt_variants)

response_records = [
    {
        "id": response.case_id,
        "variante_prompt": response.prompt_variant,
        "respuesta_cruda": response.raw_response,
        "respuesta_normalizada": response.normalized_answer,
        "evidencia_declarada": ", ".join(response.declared_evidence),
        "confianza_lingüística": response.confidence_label
    }
    for response in responses
]

if pd is not None:
    display(pd.DataFrame(response_records))
else:
    for record in response_records:
        print(json.dumps(record, ensure_ascii=False, indent=2))


### **Parte 5: Verificación de grounding**

#### **Criterios operativos**

Una respuesta está mejor grounded cuando:

1. Menciona objetos presentes.
2. Usa evidencia visible o textual esperada.
3. No inventa objetos ausentes.
4. No contradice la descripción visual.
5. Reconoce incertidumbre cuando la evidencia es insuficiente.

La verificación automática de grounding es parcial. Sirve como filtro inicial, pero no reemplaza una auditoría humana.


In [ ]:
@dataclass
class GroundingAudit:
    """Resultado de auditoría automática parcial."""
    case_id: str
    prompt_variant: str
    is_correct: bool
    mentions_expected_evidence: bool
    mentions_absent_object: bool
    is_hallucination: bool
    is_false_negative: bool
    severity: str
    error_type: str
    diagnostic: str


def get_case_by_id(cases: Sequence[GroundingCase], case_id: str):
    """Recupera un caso por identificador."""
    for case in cases:
        if case.case_id == case_id:
            return case
    raise ValueError(f"No se encontró el caso: {case_id}")


def contains_any(text: str, candidates: Iterable[str]):
    """Verifica si el texto contiene algún candidato."""
    text_lower = text.lower()
    return any(candidate.lower() in text_lower for candidate in candidates)


def classify_error(case: GroundingCase, response: ModelResponse):
    """Clasifica el tipo de error y su severidad."""
    expected = case.expected_answer
    predicted = response.normalized_answer
    raw_text = response.raw_response.lower()

    if predicted == expected:
        return "sin_error", "baja"

    if contains_any(raw_text, case.absent_objects):
        return "alucinación_visual", "alta"

    if case.task_type in ["ocr", "ocr negativo"]:
        return "error_ocr", "alta"

    if case.task_type == "relación espacial":
        return "error_espacial", "media"

    if case.task_type == "conteo":
        return "error_conteo", "media"

    if case.task_type == "pregunta negativa":
        return "error_de_grounding", "alta"

    return "error_respuesta", "media"


def audit_grounding(cases: Sequence[GroundingCase], responses: Sequence[ModelResponse]):
    """Audita grounding, alucinación y severidad de forma parcial."""
    audits = []

    for response in responses:
        case = get_case_by_id(cases, response.case_id)
        raw_text = response.raw_response.lower()

        is_correct = response.normalized_answer == case.expected_answer
        mentions_expected_evidence = contains_any(raw_text, case.expected_evidence)
        mentions_absent_object = contains_any(raw_text, case.absent_objects)

        is_yes_no_case = case.expected_answer in ["sí", "no"]
        is_hallucination = (
            is_yes_no_case
            and case.expected_answer == "no"
            and response.normalized_answer == "sí"
        ) or mentions_absent_object

        is_false_negative = (
            is_yes_no_case
            and case.expected_answer == "sí"
            and response.normalized_answer == "no"
        )

        error_type, severity = classify_error(case, response)

        if is_correct and mentions_expected_evidence:
            diagnostic = "Respuesta correcta con evidencia esperada."
        elif is_correct:
            diagnostic = "Respuesta correcta con evidencia débil o no declarada."
        elif is_hallucination:
            diagnostic = "La respuesta inventa evidencia o afirma presencia no respaldada."
        else:
            diagnostic = "La respuesta no coincide con la anotación esperada."

        audits.append(GroundingAudit(
            case_id=response.case_id,
            prompt_variant=response.prompt_variant,
            is_correct=is_correct,
            mentions_expected_evidence=mentions_expected_evidence,
            mentions_absent_object=mentions_absent_object,
            is_hallucination=is_hallucination,
            is_false_negative=is_false_negative,
            severity=severity,
            error_type=error_type,
            diagnostic=diagnostic
        ))

    return audits


audits = audit_grounding(cases, responses)

audit_records = [
    {
        "id": audit.case_id,
        "variante_prompt": audit.prompt_variant,
        "correcta": audit.is_correct,
        "menciona_evidencia": audit.mentions_expected_evidence,
        "menciona_objeto_ausente": audit.mentions_absent_object,
        "alucinación": audit.is_hallucination,
        "falso_negativo": audit.is_false_negative,
        "severidad": audit.severity,
        "tipo_error": audit.error_type,
        "diagnóstico": audit.diagnostic
    }
    for audit in audits
]

if pd is not None:
    display(pd.DataFrame(audit_records))
else:
    for record in audit_records:
        print(json.dumps(record, ensure_ascii=False, indent=2))


### **Parte 6: Métricas de alucinación y consistencia**

#### **Métricas usadas**

Se calculan métricas simples, interpretables y discutibles:

1. **Exactitud:** proporción de respuestas normalizadas correctas.
2. **Tasa de falsos positivos:** proporción de casos negativos respondidos como positivos.
3. **Tasa de falsos negativos:** proporción de casos positivos respondidos como negativos.
4. **Tasa de alucinación:** proporción de respuestas que inventan presencia o evidencia.
5. **Consistencia por prompt:** proporción de casos con la misma respuesta normalizada en todas las variantes.
6. **Severidad media:** promedio ordinal de severidad baja, media y alta.

Estas métricas no sustituyen una revisión humana. Su valor es ordenar la discusión.


In [ ]:
def safe_divide(numerator: float, denominator: float):
    """Divide evitando errores cuando el denominador es cero."""
    return numerator / denominator if denominator else 0.0


def compute_metrics(cases: Sequence[GroundingCase], responses: Sequence[ModelResponse], audits: Sequence[GroundingAudit]):
    """Calcula métricas agregadas para el protocolo."""
    total = len(audits)
    correct = sum(1 for audit in audits if audit.is_correct)
    hallucinations = sum(1 for audit in audits if audit.is_hallucination)
    false_negatives = sum(1 for audit in audits if audit.is_false_negative)

    negative_yes_no = 0
    false_positives = 0

    for response in responses:
        case = get_case_by_id(cases, response.case_id)
        if case.expected_answer == "no":
            negative_yes_no += 1
            if response.normalized_answer == "sí":
                false_positives += 1

    positive_yes_no = 0
    positive_false_negatives = 0

    for response in responses:
        case = get_case_by_id(cases, response.case_id)
        if case.expected_answer == "sí":
            positive_yes_no += 1
            if response.normalized_answer == "no":
                positive_false_negatives += 1

    severity_map = {"baja": 1, "media": 2, "alta": 3}
    severity_values = [severity_map[audit.severity] for audit in audits]

    consistency_scores = []
    for case in cases:
        answers = [
            response.normalized_answer
            for response in responses
            if response.case_id == case.case_id
        ]
        unique_answers = set(answers)
        consistency_scores.append(1.0 if len(unique_answers) == 1 else 0.0)

    return {
        "exactitud": safe_divide(correct, total),
        "tasa_falsos_positivos": safe_divide(false_positives, negative_yes_no),
        "tasa_falsos_negativos": safe_divide(positive_false_negatives, positive_yes_no),
        "tasa_alucinacion": safe_divide(hallucinations, total),
        "consistencia_por_prompt": mean(consistency_scores),
        "severidad_media": mean(severity_values),
        "total_respuestas": float(total),
        "total_alucinaciones": float(hallucinations),
        "total_falsos_negativos": float(false_negatives)
    }


metrics = compute_metrics(cases, responses, audits)

for key, value in metrics.items():
    if key.startswith("total"):
        print(f"{key}: {int(value)}")
    else:
        print(f"{key}: {value:.3f}")

metrics_path = OUTPUT_DIR / "metricas_grounding.json"
with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, ensure_ascii=False)

print(f"Métricas guardadas en: {metrics_path}")


### **Parte 7: Consistencia entre variantes de prompt**

#### **Interpretación**

La consistencia no garantiza verdad, pero la inconsistencia es una señal de fragilidad. Si una respuesta cambia al modificar levemente el prompt, el sistema puede estar dependiendo de patrones lingüísticos más que de evidencia visual.

En esta sección se revisan los casos donde las respuestas normalizadas cambian entre prompt directo, prompt con evidencia y prompt cauteloso.


In [ ]:
def compute_prompt_consistency(cases: Sequence[GroundingCase], responses: Sequence[ModelResponse]):
    """Calcula consistencia por caso entre variantes de prompt."""
    records = []

    for case in cases:
        case_responses = [
            response for response in responses
            if response.case_id == case.case_id
        ]

        answers_by_prompt = {
            response.prompt_variant: response.normalized_answer
            for response in case_responses
        }

        unique_answers = set(answers_by_prompt.values())
        is_consistent = len(unique_answers) == 1

        records.append({
            "id": case.case_id,
            "tipo_tarea": case.task_type,
            "respuesta_esperada": case.expected_answer,
            "directo": answers_by_prompt.get("directo"),
            "con_evidencia": answers_by_prompt.get("con_evidencia"),
            "cauteloso": answers_by_prompt.get("cauteloso"),
            "consistente": is_consistent
        })

    return records


consistency_records = compute_prompt_consistency(cases, responses)

if pd is not None:
    display(pd.DataFrame(consistency_records))
else:
    for record in consistency_records:
        print(json.dumps(record, ensure_ascii=False, indent=2))


### **Parte 8: Matriz de errores**

#### **Taxonomía usada**

La matriz distingue errores por origen probable:

1. Error perceptual.
2. Error de OCR.
3. Error espacial.
4. Error de conteo.
5. Error de conocimiento.
6. Error de grounding.
7. Alucinación visual.
8. Error de consistencia.
9. Error de prompt.
10. Error de evaluación.

La categoría no debe interpretarse como verdad absoluta. Es una hipótesis de diagnóstico que debe contrastarse con evidencia.


In [ ]:
def suggest_mitigation(error_type: str):
    """Sugiere una mitigación inicial según tipo de error."""
    suggestions = {
        "alucinación_visual": "Agregar preguntas negativas y verificación explícita de objetos ausentes.",
        "error_ocr": "Usar OCR auxiliar y comparar la respuesta contra texto detectado.",
        "error_espacial": "Agregar ejemplos con relaciones espaciales y pedir evidencia localizada.",
        "error_conteo": "Separar detección de instancias y razonamiento numérico.",
        "error_de_grounding": "Exigir que la respuesta cite evidencia visible antes de concluir.",
        "error_respuesta": "Revisar prompt, normalización y anotación esperada."
    }
    return suggestions.get(error_type, "Revisar manualmente el caso y la anotación.")


def build_error_matrix(cases: Sequence[GroundingCase], responses: Sequence[ModelResponse], audits: Sequence[GroundingAudit]):
    """Construye una matriz de errores para revisión humana."""
    rows = []

    for response, audit in zip(responses, audits):
        case = get_case_by_id(cases, response.case_id)

        if audit.error_type == "sin_error":
            continue

        rows.append({
            "id": case.case_id,
            "tipo_tarea": case.task_type,
            "variante_prompt": response.prompt_variant,
            "pregunta": case.question,
            "respuesta_esperada": case.expected_answer,
            "respuesta_modelo": response.raw_response,
            "tipo_error": audit.error_type,
            "severidad": audit.severity,
            "evidencia_esperada": ", ".join(case.expected_evidence),
            "diagnóstico": audit.diagnostic,
            "mitigación_inicial": suggest_mitigation(audit.error_type)
        })

    return rows


error_matrix = build_error_matrix(cases, responses, audits)

if pd is not None:
    display(pd.DataFrame(error_matrix))
else:
    for row in error_matrix:
        print(json.dumps(row, ensure_ascii=False, indent=2))


### **Parte 9: Visualización simple de resultados**

#### **Uso de gráficos**

Los gráficos son auxiliares. No reemplazan la matriz de errores ni la discusión de casos críticos.

Se recomienda observar:

1. Distribución de tipos de error.
2. Distribución de severidad.
3. Métricas agregadas principales.


In [ ]:
def count_by_key(records: Sequence[Dict[str, object]], key: str):
    """Cuenta ocurrencias por una clave."""
    counts: Dict[str, int] = {}
    for record in records:
        value = str(record[key])
        counts[value] = counts.get(value, 0) + 1
    return counts


if plt is None:
    print("Matplotlib no está instalado. Se omiten gráficos.")
else:
    error_counts = count_by_key(error_matrix, "tipo_error")

    plt.figure()
    plt.bar(list(error_counts.keys()), list(error_counts.values()))
    plt.title("Distribución de tipos de error")
    plt.xlabel("Tipo de error")
    plt.ylabel("Frecuencia")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    severity_counts = count_by_key(error_matrix, "severidad")

    plt.figure()
    plt.bar(list(severity_counts.keys()), list(severity_counts.values()))
    plt.title("Distribución de severidad")
    plt.xlabel("Severidad")
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()


### **Parte 10: Auditoría humana**

#### **Selección de casos críticos**

La auditoría humana debe revisar pocos casos, pero con detalle. Se recomienda seleccionar:

1. Casos con severidad alta.
2. Casos donde el modelo inventó evidencia.
3. Casos con inconsistencia entre variantes de prompt.
4. Casos donde la respuesta fue correcta pero sin evidencia verificable.
5. Casos donde el protocolo automático pudo equivocarse.

La auditoría debe indicar evidencia visual, diagnóstico y posible mitigación.


In [ ]:
def select_critical_cases(error_matrix: Sequence[Dict[str, object]], limit: int = 5):
    """Selecciona casos críticos para auditoría humana."""
    severity_order = {"alta": 3, "media": 2, "baja": 1}

    sorted_errors = sorted(
        error_matrix,
        key=lambda row: severity_order.get(str(row["severidad"]), 0),
        reverse=True
    )

    return sorted_errors[:limit]


critical_cases = select_critical_cases(error_matrix)

audit_template = []
for row in critical_cases:
    audit_template.append({
        "id": row["id"],
        "variante_prompt": row["variante_prompt"],
        "tipo_error": row["tipo_error"],
        "severidad": row["severidad"],
        "evidencia_visual_revisada": "",
        "diagnóstico_humano": "",
        "mitigación_propuesta": row["mitigación_inicial"],
        "observación": ""
    })

if pd is not None:
    display(pd.DataFrame(audit_template))
else:
    for row in audit_template:
        print(json.dumps(row, ensure_ascii=False, indent=2))


### **Parte 11: Limitaciones del sondeo automático**

#### **Discusión metodológica**

El protocolo de este cuaderno es útil para organizar la evaluación, pero tiene limitaciones importantes:

1. Usa descripciones estructuradas en lugar de imágenes reales.
2. El simulador no reemplaza el comportamiento de un VLM real.
3. La normalización de respuestas puede introducir errores.
4. La detección de evidencia mediante coincidencias de texto es incompleta.
5. La severidad se estima con reglas simples.
6. La consistencia por prompt no prueba grounding.
7. Una auditoría humana sigue siendo necesaria.

En un experimento real, se debería agregar una fuente independiente de verdad visual, una política de anotación, revisión interanotador y registro de prompts.


In [ ]:
def save_table(records: Sequence[Dict[str, object]], path: Path):
    """Guarda una tabla en CSV o JSON según disponibilidad."""
    if pd is not None:
        pd.DataFrame(records).to_csv(path, index=False, encoding="utf-8")
    else:
        json_path = path.with_suffix(".json")
        with json_path.open("w", encoding="utf-8") as file:
            json.dump(list(records), file, indent=2, ensure_ascii=False)
        return json_path
    return path


predictions_path = save_table(response_records, OUTPUT_DIR / "predicciones.csv")
audit_path = save_table(audit_records, OUTPUT_DIR / "auditoria_automatica.csv")
errors_path = save_table(error_matrix, OUTPUT_DIR / "matriz_errores.csv")
human_audit_path = save_table(audit_template, OUTPUT_DIR / "plantilla_auditoria_humana.csv")

print("Archivos generados:")
print(predictions_path)
print(audit_path)
print(errors_path)
print(human_audit_path)


### **Parte 12: Mini-informe**

#### **Estructura sugerida**

El mini-informe debe tener una extensión breve y centrarse en evidencia.

Debe incluir:

1. Pregunta experimental.
2. Modelo o simulador usado.
3. Datos y criterios de selección.
4. Métricas calculadas.
5. Matriz de errores.
6. Casos críticos.
7. Limitaciones del protocolo.
8. Conclusión responsable.

No se debe afirmar confiabilidad general si el experimento solo cubre un conjunto pequeño de sondeo.


In [ ]:
def build_report_template(metrics: Dict[str, float], critical_cases: Sequence[Dict[str, object]]):
    """Construye una plantilla de mini-informe en Markdown."""
    critical_summary = "\n".join(
        [
            f"- {row['id']} con error {row['tipo_error']} y severidad {row['severidad']}."
            for row in critical_cases
        ]
    )

    return f"""### **Mini-informe de Semana 10**

#### **Pregunta experimental**

¿Qué tan grounded, consistente y libre de alucinaciones es el sistema evaluado en un conjunto pequeño de sondeo multimodal?

#### **Modelo o simulador usado**

Simulador VLM reproducible usado para demostrar el protocolo sin GPU.

#### **Datos**

Conjunto pequeño de casos con objetos, texto incrustado, relaciones espaciales, conteo, preguntas negativas y distractores.

#### **Métricas principales**

- Exactitud: {metrics['exactitud']:.3f}
- Tasa de falsos positivos: {metrics['tasa_falsos_positivos']:.3f}
- Tasa de falsos negativos: {metrics['tasa_falsos_negativos']:.3f}
- Tasa de alucinación: {metrics['tasa_alucinacion']:.3f}
- Consistencia por prompt: {metrics['consistencia_por_prompt']:.3f}
- Severidad media: {metrics['severidad_media']:.3f}

#### **Casos críticos**

{critical_summary}

#### **Limitaciones**

El protocolo usa descripciones estructuradas y respuestas simuladas. La verificación automática de evidencia es parcial y debe complementarse con revisión humana.

#### **Conclusión responsable**

Los resultados permiten discutir patrones de grounding y alucinación dentro del conjunto de sondeo, pero no permiten afirmar confiabilidad general del sistema. Para sostener una afirmación más fuerte se requiere evaluación con imágenes reales, anotaciones independientes, revisión humana y comparación con un VLM real.
"""


report_text = build_report_template(metrics, critical_cases)
report_path = OUTPUT_DIR / "mini_informe_semana10.md"

with report_path.open("w", encoding="utf-8") as file:
    file.write(report_text)

print(report_text)
print(f"Mini-informe guardado en: {report_path}")


### **Parte 13: Extensión avanzada con VLM real**

#### **Uso opcional**

La versión básica del cuaderno no descarga modelos. Para un trabajo integrador, se puede reemplazar el simulador por un VLM real.

La extensión avanzada debería documentar:

1. Nombre del modelo.
2. Versión, commit o checkpoint.
3. Parámetros de generación.
4. Temperatura y estrategia de decodificación.
5. Hardware.
6. Prompts usados.
7. Fuente independiente de anotaciones.
8. Criterios de parsing de respuestas.

También se puede agregar un detector externo como referencia visual, pero debe discutirse que ese detector no es un oráculo perfecto.


In [ ]:
class RealVlmAdapter:
    """Adaptador opcional para conectar un VLM real."""

    def __init__(self, model_name: str):
        """Inicializa el adaptador sin cargar el modelo por defecto."""
        self.model_name = model_name
        self.model = None
        self.processor = None

    def load(self):
        """Carga el modelo real si el entorno lo permite."""
        raise NotImplementedError(
            "Esta función debe implementarse con transformers u otra biblioteca VLM."
        )

    def generate_response(self, image_path: Path, question: str, prompt_variant: str):
        """Genera una respuesta real a partir de imagen y pregunta."""
        raise NotImplementedError(
            "Reemplace esta función por la inferencia del modelo seleccionado."
        )


print("Extensión avanzada preparada como interfaz. No se ejecuta por defecto.")


### **Actividad de cierre**


Responde en un texto breve:

1. ¿Qué diferencia se observó entre exactitud y grounding?
2. ¿Qué errores tuvieron mayor severidad?
3. ¿Qué variante de prompt fue más estable?
4. ¿Qué tipo de alucinación apareció con mayor claridad?
5. ¿Qué parte del protocolo automático requiere revisión humana?
6. ¿Qué afirmación no sería válida con este experimento?
7. ¿Qué cambiaría para convertir este sondeo en una evaluación de investigación?.


In [ ]:
## Tus respuestas